In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.fact_predios")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.fact_predios (
    pk STRING,
    matricula STRING,
    orip INT,
    divipola INT,
    departamento_id INT,
    municipio_id INT,
    tipo_predio_id INT,
    categoria_ruralidad_id INT,
    natujur_id INT,
    num_anotacion INT,
    dinamica_2024 INT,
    documento_justificativo STRING,
    count_a INT,
    count_de INT,
    predios_nuevos INT,
    tiene_valor BOOLEAN,
    tiene_mas_de_un_valor BOOLEAN,
    estado_folio STRING,
    valor DOUBLE
)
USING DELTA
""")

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# 1. Leer la tabla Silver a Pandas (Capa de transformación)
# Extraemos los datos de la tabla que acabas de crear en SQL
df = spark.table("workspace.silver.predios_registro").toPandas()

# 2. Reindexar para asegurar que el esquema sea consistente
# Si alguna columna falta en el origen, se creará como NaN (nula)
fact = df.reindex(
    columns=[
        "pk", "matricula","fecha_radica", 
        "orip", "divipola", "departamento_id", "municipio_id", 
        "tipo_predio_id", "categoria_ruralidad_id", "num_anotacion", 
        "dinamica_2024", "natujur_id",
        "documento_justificativo", "count_a", "count_de", 
        "predios_nuevos", "tiene_valor", "tiene_mas_de_un_valor", 
        "estado_folio", "valor"
    ]
)

# 3. Procesamiento de Fechas
# Convertimos 'fecha_radica' a formato datetime de Pandas para extraer la llave
dates = pd.to_datetime(fact["fecha_radica"], errors="coerce")

# 4. Crear columnas derivadas y limpieza final
fact = (
    fact.assign(
        # Creamos la date_key (YYYYMMDD) como Int64 para optimizar JOINs en SQL
        fecha_radica_id=dates.dt.strftime("%Y%m%d").astype("Int64"),
        # Aquí puedes agregar otras lógicas, por ejemplo un ID de sistema si es necesario
        source_system=pd.Series(["REGISTRO_PREDIAL"] * len(fact))
    )
    .reindex(
        columns=[
            "pk",
            "matricula",
            "fecha_radica_id",
            "departamento_id",
            "municipio_id",
            "tipo_predio_id", 
            "categoria_ruralidad_id",
            "natujur_id",
            "orip",
            "divipola",
            "num_anotacion", 
            "dinamica_2024",
            "documento_justificativo",
            "count_a",
            "count_de", 
            "predios_nuevos",
            "tiene_valor",
            "tiene_mas_de_un_valor",
            "estado_folio",
            "valor"
        ]
    )
    # Calidad de datos: Eliminamos filas sin PK (Primary Key) y duplicados
    .dropna(subset=["pk"])
    .drop_duplicates(subset=["pk"])
)

# 5. Convertir de regreso a Spark para guardar en Delta
# Esto devuelve los datos al entorno distribuido de Databricks
df_spark = spark.createDataFrame(fact)

# Opcional: Escribir a una tabla final Gold o actualizar la Silver
# df_spark.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.reporte_predios")

In [0]:

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.fact_predios")